In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

from splitflow import WindDataProcessing
from splitflow import opto_tracking
from splitflow import FeatureFunctions
from datetime import date

# Final (Fig. 4) casting / circling generators that are not in Code/: generate_synthetic_data.py in
# FigureGeneration/VAE_pipeline. It patches the module it imports as `opto_tracking`, so point that name at
# splitflow.opto_tracking first.
sys.modules.setdefault('opto_tracking', opto_tracking)
sys.path.insert(0, str((Path.cwd() / '../../FigureGeneration/VAE_pipeline').resolve()))
import generate_synthetic_data as GS
ORIG_CAST_AND_SURGE = opto_tracking.SurgingAgent.cast_and_surge   # the original casting (restored before 'casting' runs)

# Final casting class = half each of two generators (generate_synthetic_data.py CASTING / CASTING_T / CASTING_T_CLOCK;
# generate_cfd_data.py GENERATORS):
#   'casting'    original cast_and_surge (periodic sign(sin) bias clock), NOISE 1
#   'casting_t'  bias half-periods drawn N(0.6, 0.2) s (clipped 0.2-1.5 s), NOISE 0.5, slow along-wind OU (sd 0.25, tau 0.5 s)
def use_casting(component):
    if component == 'casting':
        opto_tracking.SurgingAgent.cast_and_surge = ORIG_CAST_AND_SURGE
    else:
        GS.use_clock(half_mean=0.6, half_sd=0.2, slow_x_sd=0.25, slow_x_tau=0.5)
CASTING_NOISE = {'casting': 1, 'casting_t': 0.5}

# Final smooth circling 'smooth_t4p' (generate_synthetic_data.py SMOOTH / SMOOTH_MOD): sink_and_circle with the circling
# amplitude x exp(OU, sd 0.3, tau 0.4 s), a random circling phase per agent, left / right 50/50
def use_smooth_circling():
    GS.use_amp_mod(sd=0.3, tau=0.4, random_phase=True)
    opto_tracking.SurgingAgent.both_ways = True


today = date.today()
print(today)

In [228]:
today

datetime.date(2026, 1, 15)

In [227]:
today = date(2026, 1, 15)

In [75]:
from importlib import reload

In [134]:
reload(opto_tracking)

<module 'opto_tracking' from '/media/beast/extradrive4/jaleesa/ShearFlowPaper/Simulations/opto_tracking.py'>

# Run simulations

## Casting - Laminar (constant wind)

In [ ]:

#x = np.random.choice(12000)
SEEDS = {'casting': 2177, 'casting_t': 3001}   # seeds of the final training set (generate_synthetic_data.py)

#params 
BEHAVIOR='cast_and_surge'
TAU = 0.25
TAU_SURGE = 0.02
BIAS= 0.7
SURGE_AMP=0.42
N_TRAJS=500   # per casting generator (1000 in all)
REJECT_THRESH=0
RADIUS=0
OMEGA=0

sims_df= []
cast_dfs = []

for comp in ['casting', 'casting_t']:
    use_casting(comp)
    x, NOISE = SEEDS[comp], CASTING_NOISE[comp]
    print(comp, 'seed:', x)

    sim_results = opto_tracking.run_sims(BEHAVIOR=BEHAVIOR,
        wind_condition='constant',
        SEED=x, N_TRAJS=N_TRAJS,
        DURATION=3.21,
        DT=0.01,
        TAU=TAU, NOISE=NOISE, BIAS=BIAS, 
        SURGE_AMP=SURGE_AMP,
        TAU_SURGE=TAU_SURGE,                               
        REJECT_THRESH=0,# RADIUS=0, OMEGA=0,                                 
        CONSTANT_NOISE=True,
        BOUNDS=[(-0.5, 0.5), (-0.25, 0.25), (0, 0.5)], wind_data=None,  vector_field=None, wind_positions=None, wind_data_type='CFD')

    cast_dfs.append(opto_tracking.convert_and_save_sims(sim_results, 'sim_results/SimResults_tau_' + str(TAU_SURGE) + '_noise_' + str(NOISE) + '_bias_' + str(BIAS) + '_surgeamp_' + str(SURGE_AMP) +'_tau_' + str(TAU) + '_tausurge_' + str(TAU_SURGE)  + '_n_' + str(N_TRAJS) + '_seed_'+ str(x) + '_' , str(today) + '_'+ comp + '_constantwind' , remove_wall_hits=False, save=True).assign(component=comp))

df = pd.concat(cast_dfs, ignore_index=True)


In [234]:
#calculate position normalized to point when odor turned off
df=FeatureFunctions.get_normed_pos_groupby(df)
df.to_hdf('sim_results/casting_sim_constwind_n1000.hdf',key='n')

/media/beast/extradrive4/jaleesa/ShearFlowPaper/Simulations/FeatureFunctions.py:314: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('obj_id_unique', as_index=False).apply(lambda df: df.assign(normed_x_680 = lambda d: d.x - d[d['time stamp']==680].x.iloc[0] ))
/media/beast/extradrive4/jaleesa/ShearFlowPaper/Simulations/FeatureFunctions.py:315: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupb

# Circling - Still air (No Wind)

In [ ]:

#x = np.random.choice(12000)
x= 4301   # seed of the final smooth circling (generate_synthetic_data.py)
print('seed:',x)

sims_df= []

#params (final smooth circling smooth_t4p: generate_synthetic_data.SMOOTH)
BEHAVIOR='sink_and_circle'
TAU = 0.25
TAU_SURGE= 0.25
NOISE = 0.3
BIAS= 1.8
SURGE_AMP=0
RADIUS=0.3
OMEGA=4.05 #6
REJECT_THRESH=0
N_TRAJS=1000

use_smooth_circling()

sim_results = opto_tracking.run_sims(BEHAVIOR=BEHAVIOR,
    wind_condition='constant',
    SEED=x, N_TRAJS=N_TRAJS,
    DURATION=3.21,
    DT=0.01,
    TAU=TAU, NOISE=NOISE, BIAS=BIAS, #.6 for flow
    SURGE_AMP=SURGE_AMP,
    TAU_SURGE=TAU_SURGE,                               
    REJECT_THRESH=REJECT_THRESH,RADIUS=RADIUS, OMEGA=OMEGA,                                 
    CONSTANT_NOISE=True,
    BOUNDS=[(-0.5, 0.5), (-0.25, 0.25), (0, 0.5)], wind_data=None,  vector_field=None, wind_positions=None, wind_data_type='CFD')


df = opto_tracking.convert_and_save_sims(sim_results, 'sim_results/SimResults_tau_' + str(TAU_SURGE) + '_noise_' + str(NOISE) + '_bias_' + str(BIAS) + '_surgeamp_' + str(SURGE_AMP) +'_tau_' + str(TAU) + '_tausurge_' + str(TAU_SURGE) + '_rejectthresh_'+ str(REJECT_THRESH)  + '_n_' + str(N_TRAJS) + '_seed_'+ str(x) + '_' , str(today) + '_smooth_t4p_constantwind' , remove_wall_hits=False, save=True)


In [166]:
#calculate position normalized to point when odor turned off
df=FeatureFunctions.get_normed_pos_groupby(df)
df.to_hdf('sim_results/circle_sim_nowind_n1000.hdf',key='n')

/media/beast/extradrive4/jaleesa/ShearFlowPaper/Simulations/FeatureFunctions.py:314: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('obj_id_unique', as_index=False).apply(lambda df: df.assign(normed_x_680 = lambda d: d.x - d[d['time stamp']==680].x.iloc[0] ))
/media/beast/extradrive4/jaleesa/ShearFlowPaper/Simulations/FeatureFunctions.py:315: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupb

# Circling with CFD wind

In [ ]:
directories = ['vel_1','vel_2','vel_3', 'vel_4', 'vel_5', 'vel_6',]
#x = np.random.choice(12000)
x=5613
print('seed:',x)
#sims_df= []

#params (final smooth circling smooth_t4p: generate_synthetic_data.SMOOTH; REJECT_THRESH 0.3 as generate_cfd_data)
BEHAVIOR='sink_and_circle'
TAU = 0.25
TAU_SURGE= 0.25
NOISE = 0.3
BIAS= 1.8
REJECT_THRESH = 0.3
SURGE_AMP=0
RADIUS = 0.3
OMEGA  = 4.05
N_TRAJS=1000

use_smooth_circling()

for i, directory in enumerate(directories):
    
    print('loading ~3 seconds of CFD data:')
    
    ## makes sure that any prior wind data is deleted from memory before loading new data
    try:
        del dfs
    except NameError:
        pass
    
    dfs = WindDataProcessing.load_wind_data(directory, filetype='parquet', datatype='CFD')
    
    pos = [dfs[0].x.unique(),dfs[0].y.unique(),dfs[0].z.unique()]
    
    sim_results = opto_tracking.run_sims(
        SEED=x, BEHAVIOR=BEHAVIOR, N_TRAJS=N_TRAJS,
        DURATION=3.21,
        DT=0.01,
        TAU=TAU, NOISE=NOISE, BIAS=BIAS, 
        SURGE_AMP=SURGE_AMP,
        TAU_SURGE=TAU_SURGE, REJECT_THRESH=REJECT_THRESH,
        RADIUS=RADIUS, OMEGA = OMEGA,
        CONSTANT_NOISE=True,
        wind_condition='data',
        BOUNDS=[(-0.5, 0.5), (-0.25, 0.25), (0, 0.5)], wind_data=dfs,  vector_field=dfs, wind_positions=pos, wind_data_type='CFD')

    
    df = opto_tracking.convert_and_save_sims(sim_results, 'sim_results/SimResults_tau_' + str(TAU_SURGE) + '_noise_' + str(NOISE) + '_bias_' + str(BIAS) + '_surgeamp_' + str(SURGE_AMP) + '_tausurge_' + str(TAU_SURGE) + '_reject_' + str(REJECT_THRESH) + '_n_' + str(N_TRAJS) + '_seed_'+ str(x) + '_' , str(today) + '_smooth_t4p_' + directory,remove_wall_hits=False, save=True)
    sims_df.append(df)


In [149]:
alldfs = pd.concat(sims_df)
#calculate position normalized to point when odor turned off
alldfs=FeatureFunctions.get_normed_pos_groupby(alldfs)
alldfs.to_hdf('sim_results/cfd_sim_results_merged_circling_'+ str(today), key='wind')

/media/beast/extradrive4/jaleesa/ShearFlowPaper/Simulations/FeatureFunctions.py:314: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('obj_id_unique', as_index=False).apply(lambda df: df.assign(normed_x_680 = lambda d: d.x - d[d['time stamp']==680].x.iloc[0] ))
/media/beast/extradrive4/jaleesa/ShearFlowPaper/Simulations/FeatureFunctions.py:315: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupb

# Casting with CFD wind

In [ ]:
directories = ['vel_1',  'vel_2', 'vel_3', 'vel_4', 'vel_5', 'vel_6',]
#x = np.random.choice(12000)
SEEDS = {'casting': 5612, 'casting_t': 5614}
sims_df1= []

#params 
BEHAVIOR='cast_and_surge'
TAU = 0.25
TAU_SURGE = 0.02
BIAS= 0.7
SURGE_AMP=0.42
N_TRAJS=500   # per casting generator and condition (as generate_cfd_data)
REJECT_THRESH=0
RADIUS=0
OMEGA=0




for i, directory in enumerate(directories):
    
    print('loading ~3 seconds of CFD data:')
    
    ## makes sure that any prior wind data is deleted from memory before loading new data
    try:
        del dfs
    except NameError:
        pass
    
    dfs = WindDataProcessing.load_wind_data(directory, filetype='parquet', datatype='CFD')
    
    pos = [dfs[0].x.unique(),dfs[0].y.unique(),dfs[0].z.unique()]
    
    for comp in ['casting', 'casting_t']:          # both casting generators on each loaded condition
        use_casting(comp)
        x, NOISE = SEEDS[comp], CASTING_NOISE[comp]
        print(comp, 'seed:', x)
        sim_results = opto_tracking.run_sims(
            SEED=x, BEHAVIOR=BEHAVIOR, N_TRAJS=N_TRAJS,
            DURATION=3.21,
            DT=0.01,
            TAU=TAU, NOISE=NOISE, BIAS=BIAS, 
            SURGE_AMP=SURGE_AMP,
            TAU_SURGE=TAU_SURGE, REJECT_THRESH=REJECT_THRESH,
            RADIUS=RADIUS, OMEGA = OMEGA,
            CONSTANT_NOISE=True,
            wind_condition='data',
            BOUNDS=[(-0.5, 0.5), (-0.25, 0.25), (0, 0.5)], wind_data=dfs,  vector_field=dfs, wind_positions=pos, wind_data_type='CFD')

    
        df = opto_tracking.convert_and_save_sims(sim_results, 'sim_results/SimResults_tau_' + str(TAU_SURGE) + '_noise_' + str(NOISE) + '_bias_' + str(BIAS) + '_surgeamp_' + str(SURGE_AMP) + '_tausurge_' + str(TAU_SURGE) + '_reject_' + str(REJECT_THRESH) + '_n_' + str(N_TRAJS) + '_seed_'+ str(x) + '_' , str(today) + '_'+ comp + '_' + directory, remove_wall_hits=False, save=True)
        sims_df1.append(df.assign(component=comp))
    


In [175]:
alldfs2 = pd.concat(sims_df1)
#calculate position normalized to point when odor turned off
alldfs2=FeatureFunctions.get_normed_pos_groupby(alldfs2)
alldfs2.to_hdf('sim_results/cfd_sim_results_merged_casting_'+ str(today), key='wind')

/media/beast/extradrive4/jaleesa/ShearFlowPaper/Simulations/FeatureFunctions.py:314: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('obj_id_unique', as_index=False).apply(lambda df: df.assign(normed_x_680 = lambda d: d.x - d[d['time stamp']==680].x.iloc[0] ))
/media/beast/extradrive4/jaleesa/ShearFlowPaper/Simulations/FeatureFunctions.py:315: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupb

In [177]:
alldfs3 = pd.concat([alldfs1, alldfs2])

In [178]:
unsteady_casting = alldfs3.groupby('obj_id_unique').filter(lambda d: d.z.max()<.25).reset_index(drop=True)
steady_casting = alldfs3.groupby('obj_id_unique').filter(lambda d: d.z.min()>.25).reset_index(drop=True)


In [156]:
unsteady_circle = alldfs.groupby('obj_id_unique').filter(lambda d: d.z.max()<.25).reset_index(drop=True)
steady_circle = alldfs.groupby('obj_id_unique').filter(lambda d: d.z.min()>.25).reset_index(drop=True)


In [181]:
subsample_ids = unsteady_casting.drop_duplicates('obj_id_unique').apply(lambda x: x.sample(n=200, replace=False)).obj_id_unique.values
unsteady_casting = unsteady_casting[unsteady_casting.obj_id_unique.isin(subsample_ids)]
unsteady_casting['dataset_name'] = 'unsteady casting'

subsample_ids = steady_casting.drop_duplicates('obj_id_unique').apply(lambda x: x.sample(n=200, replace=False)).obj_id_unique.values
steady_casting= steady_casting[steady_casting.obj_id_unique.isin(subsample_ids)]
steady_casting['dataset_name'] = 'steady casting'

subsample_ids = unsteady_circle.drop_duplicates('obj_id_unique').apply(lambda x: x.sample(n=200, replace=False)).obj_id_unique.values
unsteady_circle  = unsteady_circle[unsteady_circle.obj_id_unique.isin(subsample_ids)]
unsteady_circle['dataset_name'] = 'unsteady circling'

subsample_ids = steady_circle.drop_duplicates('obj_id_unique').apply(lambda x: x.sample(n=200, replace=False)).obj_id_unique.values
steady_circle = steady_circle[steady_circle.obj_id_unique.isin(subsample_ids)]
steady_circle['dataset_name'] = 'steady circling'

/tmp/ipykernel_536107/1678434076.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  steady_circle['dataset_name'] = 'steady circling'


In [182]:
casting_simple = pd.read_hdf('sim_results/casting_sim_constwind_n1000.hdf')
circle_simple = pd.read_hdf('sim_results/circle_sim_nowind_n1000.hdf')

In [183]:
subsample_ids = casting_simple.drop_duplicates('obj_id_unique').apply(lambda x: x.sample(n=200, replace=False)).obj_id_unique.values
casting_simple = casting_simple[casting_simple.obj_id_unique.isin(subsample_ids)]
casting_simple['dataset_name'] = 'laminar casting'

subsample_ids = circle_simple.drop_duplicates('obj_id_unique').apply(lambda x: x.sample(n=200, replace=False)).obj_id_unique.values
circle_simple = circle_simple[circle_simple.obj_id_unique.isin(subsample_ids)]
circle_simple['dataset_name'] = 'still air circle'

In [184]:
bigdf=pd.concat([unsteady_casting,steady_casting,unsteady_circle,steady_circle,casting_simple,circle_simple ])

In [ ]:
bigdf.to_parquet('example_cast_circle_trajectories_cfd.parquet')